# Face Prompt Studio — Colab
Teach Stable Diffusion your face with LoRA, then generate yourself in any scene.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Total time ≈ 30 min.

In [ ]:
%cd /content
import os
if not os.path.exists('face-prompt-studio'):
    !git clone -q https://github.com/Avani19051/face-prompt-studio.git
%cd /content/face-prompt-studio
!pip -q install diffusers transformers accelerate peft safetensors face-alignment insightface onnxruntime-gpu gradio

import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU on this runtime. Go to Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.')
print('GPU:', torch.cuda.get_device_name(0))

## Steps 1–2 · Upload 10–20 photos, then detect, align and crop
Different angles, expressions and lighting. One person per photo.

In [ ]:
import os
from google.colab import files
os.makedirs('data/raw', exist_ok=True)
for name, content in files.upload().items():
    open(f'data/raw/{name}', 'wb').write(content)
!python src/face_prep.py --input data/raw --output data/aligned

In [ ]:
from PIL import Image
import glob
from src.generate import grid
crops = [Image.open(p).resize((192, 192)) for p in sorted(glob.glob('data/aligned/*.png'))]
grid(crops, cols=6)

## Steps 3–4 · Train the LoRA (~15–25 min)
A preview grid is saved every 200 steps in `outputs/lora/previews`.

In [ ]:
!python src/train_lora.py --instance_dir data/aligned --output_dir outputs/lora --max_steps 800

In [ ]:
from IPython.display import display
for p in sorted(glob.glob('outputs/lora/previews/*.png')):
    print(p); display(Image.open(p).resize((1024, 256)))

## Steps 5–7 · Generate yourself in any scene
Keep **`sks person`** in the prompt.

In [ ]:
prompts = [
    'a portrait photo of sks person as an astronaut, cinematic lighting',
    'sks person as a renaissance oil painting',
    'sks person reading a book in a cozy cafe, 35mm film photo',
]
for p in prompts:
    !python src/generate.py --lora_dir outputs/lora -p "{p}" -n 4 -o outputs/generated
for p in sorted(glob.glob('outputs/generated/*grid.png')):
    display(Image.open(p).resize((1024, 256)))

## Identity check · LoRA vs. base model
ArcFace cosine similarity to your real photos (above 0.5 ≈ same person).

In [ ]:
import sys; sys.path.insert(0, 'src')
from generate import load_pipeline, generate
base, _ = load_pipeline(None, base_model='stable-diffusion-v1-5/stable-diffusion-v1-5')
os.makedirs('outputs/baseline', exist_ok=True)
for i, p in enumerate(prompts):
    for j, im in enumerate(generate(base, p, n=4, seed=0)):
        im.save(f'outputs/baseline/{i}_{j}.png')
del base; torch.cuda.empty_cache()
!python src/evaluate_identity.py --reference_dir data/aligned --generated_dir outputs/generated --baseline_dir outputs/baseline

## Web demo
Opens a public Gradio link you can share.

In [ ]:
!python app.py --lora_dir outputs/lora --reference_dir data/aligned --share

## Save your LoRA (optional)

In [ ]:
!zip -qr my_face_lora.zip outputs/lora -x 'outputs/lora/previews/*'
files.download('my_face_lora.zip')